In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime

# -----------------------------------
# Basic configuration
# -----------------------------------

# This makes the request look like it is coming from a normal browser.
# Some sites are more likely to block requests if no User-Agent is provided.
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# Name of the output file that will store the final combined data.
OUT_CSV = "ufc_combined_enriched.csv"

# If you only want one specific event, put its exact name here.
# If you want all events, set this to None.
TARGET_EVENT_NAME = "UFC Fight Night: Emmett vs. Vallejos"

# If you want to keep only events before a certain date, set it here.
# Otherwise leave it as None.
BEFORE_DATE = None

# Small pause between requests so we do not hit the site too aggressively.
SLEEP_BETWEEN_REQUESTS = 0.75


# -----------------------------------
# Helper: download page and parse HTML
# -----------------------------------
def get_soup(url):
    """
    Sends a GET request to the given URL and returns a BeautifulSoup object.

    Why this exists:
    - avoids repeating requests.get(...) everywhere
    - gives us parsed HTML directly
    """
    r = requests.get(url, headers=HEADERS, timeout=15)
    r.raise_for_status()   # raises an error if the request failed
    return BeautifulSoup(r.text, "html.parser")


# -----------------------------------
# Helper: find a table by checking its headers
# -----------------------------------
def find_table_by_headers(soup, required_headers):
    """
    Looks through all tables on the page and returns the one whose headers
    contain all the required keywords.

    Example:
    If we want the totals table, we search for headers like
    KD, SIG, TOTAL, TD, SUB, REV, CTRL.
    """
    required_headers = [h.lower() for h in required_headers]

    for table in soup.find_all("table"):
        headers = [th.get_text(" ", strip=True).lower() for th in table.find_all("th")]

        # Check whether every required header appears somewhere in this table
        if all(any(req in h for h in headers) for req in required_headers):
            return table

    return None


# -----------------------------------
# Helper: split a UFCStats row into red-corner values and blue-corner values
# -----------------------------------
def row_red_blue_values(row):
    """
    UFCStats often stores fight table data as alternating values:
    red fighter value, blue fighter value, red fighter value, blue fighter value...

    This function separates them into two lists:
    - red values
    - blue values
    """
    ps = [p.get_text(" ", strip=True) for p in row.find_all("p")]

    # Start from index 0 and take every second value -> red fighter
    red = ps[0::2]

    # Start from index 1 and take every second value -> blue fighter
    blue = ps[1::2]

    return red, blue


# -----------------------------------
# Scrape metadata from a fighter's profile page
# -----------------------------------
def parse_fighter_profile(url):
    """
    Visits a fighter profile page and extracts:
    - height
    - reach
    - stance
    - dob

    Returns a dictionary with those values.
    """
    soup = get_soup(url)

    # Fighter info is usually inside this unordered list
    ul = soup.find("ul", class_="b-list__box-list")

    # If the profile layout is missing or changed, return empty values
    if not ul:
        return {
            "height": None,
            "reach": None,
            "stance": None,
            "dob": None
        }

    data = {}

    # Each list item contains something like:
    # Height: 5' 10"
    # Reach: 72"
    # DOB: Jan 01, 1990
    for li in ul.find_all("li", class_="b-list__box-list-item"):
        text = " ".join(li.get_text(" ", strip=True).split())

        # Split into key/value around the first colon
        if ":" in text:
            key, val = text.split(":", 1)
            data[key.strip().lower()] = val.strip()

    return {
        "height": data.get("height"),
        "reach": data.get("reach"),
        "stance": data.get("stance"),
        "dob": data.get("dob")
    }


# -----------------------------------
# Scrape completed event list
# -----------------------------------
def get_completed_events(target_event_name=None, before_date=None):
    """
    Scrapes the full completed-events page and returns a list of events.

    Each event is stored as a dictionary containing:
    - name
    - date (string)
    - date_dt (datetime object)
    - url

    Filters:
    - target_event_name: keep only one exact event name
    - before_date: keep only events before this date
    """
    url = "http://ufcstats.com/statistics/events/completed?page=all"
    soup = get_soup(url)

    event_details = []

    table = soup.find("tbody")
    if not table:
        return event_details

    for tr in table.find_all("tr"):
        a = tr.find("a", class_="b-link b-link_style_black")
        if not a:
            continue

        name = a.text.strip()

        span = tr.select_one("span")
        if not span:
            continue

        date_str = span.text.strip()
        event_date = datetime.strptime(date_str, "%B %d, %Y")

        # If a specific event name was requested, skip everything else
        if target_event_name is not None and name != target_event_name:
            continue

        # If a cutoff date was provided, skip newer events
        if before_date is not None and event_date >= before_date:
            continue

        event_details.append({
            "name": name,
            "date": date_str,
            "date_dt": event_date,
            "url": a.get("href")
        })

    return event_details


# -----------------------------------
# Scrape all fight links from one event page
# -----------------------------------
def get_fight_links(event_url):
    """
    Opens an event page and extracts links to all fights on that card.

    Returns a list of unique fight URLs.
    """
    soup = get_soup(event_url)
    table = soup.find("tbody")

    if not table:
        return []

    fight_links = []
    seen = set()

    for a in table.find_all("a", class_="b-flag"):
        href = a.get("href")

        # Only keep non-empty links and avoid duplicates
        if href and href not in seen:
            seen.add(href)
            fight_links.append(href)

    return fight_links


# -----------------------------------
# Scrape one fight page and enrich it with fighter profile data
# -----------------------------------
# def get_fight_with_profiles(fight_link, event_name, event_date, fighter_profile_cache):
#     """
#     Scrapes one fight page and returns a full row of data.

#     Includes:
#     - fight details (winner, weightclass, method, round, etc.)
#     - post-fight stats from the page
#     - event date
#     - fighter profile metadata for both fighters

#     fighter_profile_cache is used so we do not scrape the same fighter profile
#     repeatedly if that fighter appears multiple times across the dataset.
#     """
#     soup = get_soup(fight_link)

#     # winner = None
#     # draw_type = None

#     # These will store the two fighters shown on the page
#     fighter_names = []
#     fighter_urls = []

#     # -----------------------------------
#     # Get fighter names, profile links, and winner status
#     # -----------------------------------
#     for person in soup.find_all("div", class_="b-fight-details__person"):
#         status_tag = person.find("i", class_="b-fight-details__person-status")
#         name_tag = person.find("a", class_="b-link b-fight-details__person-link")

#         name = name_tag.get_text(strip=True) if name_tag else ""
#         url = name_tag.get("href") if name_tag else None
#         status = status_tag.get_text(strip=True) if status_tag else ""

#         fighter_names.append(name)
#         fighter_urls.append(url)

#         # # UFCStats uses W for winner, D for draw, NC for no contest
#         # if status == "W":
#         #     winner = name
#         # elif status in ["D", "NC"]:
#         #     draw_type = status

#     # If no winner is marked, handle draw / no contest cases
#     # if winner is None:
#     #     if draw_type == "D":
#     #         winner = "DRAW"
#     #     elif draw_type == "NC":
#     #         winner = "NO CONTEST"

#     # If for some reason fewer than 2 fighters were found, skip this fight
#     if len(fighter_names) < 2:
#         return None

#     # -----------------------------------
#     # Basic fight details
#     # -----------------------------------
#     # title_tag = soup.find("h2", class_="b-content__title")
#     # title = title_tag.get_text(" ", strip=True) if title_tag else event_name

#     # weight_class_tag = soup.find("i", class_="b-fight-details__fight-title")
#     # weight_class = (
#     #     weight_class_tag.get_text(" ", strip=True).replace("Bout", "").strip()
#     #     if weight_class_tag else ""
#     # )

#     # method = ""
#     # method_outer = soup.find("i", class_="b-fight-details__text-item_first")
#     # if method_outer:
#     #     inner = method_outer.find_all("i")
#     #     if len(inner) >= 2:
#     #         method = inner[1].text.strip()

#     # round_ = ""
#     # time_ = ""
#     # format_ = ""

#     # UFCStats stores round, time, and time format in these text items
#     # items = soup.find_all("i", class_="b-fight-details__text-item")
#     # for item in items[:3]:
#     #     label_tag = item.find("i", class_="b-fight-details__label")
#     #     if not label_tag:
#     #         continue

#     #     label = label_tag.text.strip().rstrip(":")
#     #     label_tag.extract()
#     #     value = item.text.strip()

#     #     if label == "Round":
#     #         round_ = value
#     #     elif label == "Time":
#     #         time_ = value
#     #     elif label == "Time format":
#     #         # Keep only the first part if extra text appears
#     #         format_ = value.split()[0]

#     # -----------------------------------
#     # Find the main stat tables
#     # -----------------------------------

#     # Totals table contains things like KD, sig strikes, takedowns, etc.
#     # totals_table = find_table_by_headers(
#     #     soup, ["KD", "SIG", "TOTAL", "TD", "SUB", "REV", "CTRL"]
#     # )

#     # # This table contains head/body/leg plus distance/clinch/ground breakdowns
#     # target_table = find_table_by_headers(
#     #     soup, ["HEAD", "BODY", "LEG", "DISTANCE", "CLINCH", "GROUND"]
#     # )

#     # def first_row_rb(table):
#     #     """
#     #     Helper to safely get the first row of a table and split it into
#     #     red-corner values and blue-corner values.
#     #     """
#     #     if not table:
#     #         return [], []

#     #     tbody = table.find("tbody")
#     #     if not tbody:
#     #         return [], []

#     #     row = tbody.find("tr")
#     #     if not row:
#     #         return [], []

#     #     return row_red_blue_values(row)

#     # totals_red, totals_blue = first_row_rb(totals_table)
#     # target_red, target_blue = first_row_rb(target_table)

#     # -----------------------------------
#     # Default values in case tables are missing
#     # -----------------------------------
#     KD1 = KD2 = SIG1 = SIG2 = TOT1 = TOT2 = TD1 = TD2 = ""
#     SUB1 = SUB2 = REV1 = REV2 = CTRL1 = CTRL2 = ""
#     Head1 = Head2 = Body1 = Body2 = Leg1 = Leg2 = ""
#     Distance1 = Distance2 = Clinch1 = Clinch2 = Ground1 = Ground2 = ""

#     # -----------------------------------
#     # Parse totals table values
#     # -----------------------------------
#     if len(totals_red) >= 10 and len(totals_blue) >= 10:
#         # First entry is usually fighter name, so remove it
#         totals_red = totals_red[1:]
#         totals_blue = totals_blue[1:]

#         # Order on UFCStats is usually:
#         # KD, SIG STR, SIG STR %, TOTAL STR, TD, TD %, SUB ATT, REV, CTRL
#         KD1, SIG1, _, TOT1, TD1, _, SUB1, REV1, CTRL1 = totals_red[:9]
#         KD2, SIG2, _, TOT2, TD2, _, SUB2, REV2, CTRL2 = totals_blue[:9]

#     # -----------------------------------
#     # Parse target/position breakdown values
#     # -----------------------------------
#     # if len(target_red) >= 6 and len(target_blue) >= 6:
#     #     # Last 6 values are usually:
#     #     # HEAD, BODY, LEG, DISTANCE, CLINCH, GROUND
#     #     Head1, Body1, Leg1, Distance1, Clinch1, Ground1 = target_red[-6:]
#     #     Head2, Body2, Leg2, Distance2, Clinch2, Ground2 = target_blue[-6:]

#     # -----------------------------------
#     # Scrape fighter profiles, but use cache so repeated fighters are not re-downloaded
#     # -----------------------------------
#     f1_url = fighter_urls[0]
#     f2_url = fighter_urls[1]

#     if f1_url and f1_url not in fighter_profile_cache:
#         fighter_profile_cache[f1_url] = parse_fighter_profile(f1_url)
#         time.sleep(SLEEP_BETWEEN_REQUESTS)

#     if f2_url and f2_url not in fighter_profile_cache:
#         fighter_profile_cache[f2_url] = parse_fighter_profile(f2_url)
#         time.sleep(SLEEP_BETWEEN_REQUESTS)

#     f1_meta = fighter_profile_cache.get(f1_url, {})
#     f2_meta = fighter_profile_cache.get(f2_url, {})

#     # -----------------------------------
#     # Return one fully enriched fight row
#     # -----------------------------------
#     return {
#         "Event": event_name,
#         "Fighter1": fighter_names[0],
#         "Fighter2": fighter_names[1],
#         # "Winner": winner,
#         # "Weightclass": weight_class,
#         # "Method": method,
#         # "Round": round_,
#         # "Time": time_,
#         # "Format": format_,
#         # "KD1": KD1,
#         # "KD2": KD2,
#         # "SIG_STR1": SIG1,
#         # "SIG_STR2": SIG2,
#         # "TOTAL_STR1": TOT1,
#         # "TOTAL_STR2": TOT2,
#         # "TD1": TD1,
#         # "TD2": TD2,
#         # "SUB_ATT1": SUB1,
#         # "SUB_ATT2": SUB2,
#         # "REV1": REV1,
#         # "REV2": REV2,
#         # "CTRL1": CTRL1,
#         # "CTRL2": CTRL2,
#         # "Head1": Head1,
#         # "Head2": Head2,
#         # "Body1": Body1,
#         # "Body2": Body2,
#         # "Leg1": Leg1,
#         # "Leg2": Leg2,
#         # "Distance1": Distance1,
#         # "Distance2": Distance2,
#         # "Clinch1": Clinch1,
#         # "Clinch2": Clinch2,
#         # "Ground1": Ground1,
#         # "Ground2": Ground2,
#         # "event_date": event_date,
#         "f1_dob": f1_meta.get("dob"),
#         "f1_height": f1_meta.get("height"),
#         "f1_reach": f1_meta.get("reach"),
#         "f1_stance": f1_meta.get("stance"),
#         "f2_dob": f2_meta.get("dob"),
#         "f2_height": f2_meta.get("height"),
#         "f2_reach": f2_meta.get("reach"),
#         "f2_stance": f2_meta.get("stance"),
#         "fight_url": fight_link,
#         "f1_url": f1_url,
#         "f2_url": f2_url,
#     }

def get_fight_with_profiles(fight_link, event_name, event_date, fighter_profile_cache):
    soup = get_soup(fight_link)

    fighter_names = []
    fighter_urls = []

    for person in soup.find_all("div", class_="b-fight-details__person"):
        name_tag = person.find("a", class_="b-link b-fight-details__person-link")

        name = name_tag.get_text(strip=True) if name_tag else ""
        url = name_tag.get("href") if name_tag else None

        fighter_names.append(name)
        fighter_urls.append(url)

    if len(fighter_names) < 2:
        return None

    f1_url = fighter_urls[0]
    f2_url = fighter_urls[1]

    if f1_url and f1_url not in fighter_profile_cache:
        fighter_profile_cache[f1_url] = parse_fighter_profile(f1_url)
        time.sleep(SLEEP_BETWEEN_REQUESTS)

    if f2_url and f2_url not in fighter_profile_cache:
        fighter_profile_cache[f2_url] = parse_fighter_profile(f2_url)
        time.sleep(SLEEP_BETWEEN_REQUESTS)

    f1_meta = fighter_profile_cache.get(f1_url, {})
    f2_meta = fighter_profile_cache.get(f2_url, {})

    return {
        "Event": event_name,
        "event_date": event_date,
        "Fighter1": fighter_names[0],
        "Fighter2": fighter_names[1],
        "f1_dob": f1_meta.get("dob"),
        "f1_height": f1_meta.get("height"),
        "f1_reach": f1_meta.get("reach"),
        "f1_stance": f1_meta.get("stance"),
        "f2_dob": f2_meta.get("dob"),
        "f2_height": f2_meta.get("height"),
        "f2_reach": f2_meta.get("reach"),
        "f2_stance": f2_meta.get("stance"),
        "fight_url": fight_link,
        "f1_url": f1_url,
        "f2_url": f2_url,
    }
# -----------------------------------
# Main pipeline: scrape all selected events and all their fights
# -----------------------------------
def scrape_and_enrich(target_event_name=None, before_date=None, out_csv="ufc_combined_enriched.csv"):
    """
    Main function for the whole workflow.

    Steps:
    1. Get matching events
    2. For each event, get all fight links
    3. For each fight, scrape fight stats + fighter metadata
    4. Build a DataFrame
    5. Save to CSV
    """
    events = get_completed_events(
        target_event_name=target_event_name,
        before_date=before_date
    )

    print(f"Found {len(events)} matching events")

    # Cache fighter profile results so repeated fighters do not cause repeat scraping
    fighter_profile_cache = {}

    # Store all fight rows here before converting to DataFrame
    rows = []

    for event in events:
        print(f"\nScraping event: {event['name']} | {event['date']}")

        fight_links = get_fight_links(event["url"])
        print(f"  Fight count: {len(fight_links)}")

        for i, fight_link in enumerate(fight_links, start=1):
            try:
                print(f"    [{i}/{len(fight_links)}] {fight_link}")

                row = get_fight_with_profiles(
                    fight_link=fight_link,
                    event_name=event["name"],
                    event_date=event["date"],
                    fighter_profile_cache=fighter_profile_cache
                )

                if row is not None:
                    rows.append(row)

                # pause after each fight page
                time.sleep(SLEEP_BETWEEN_REQUESTS)

            except Exception as e:
                print(f"    Error scraping fight: {fight_link}")
                print(f"    {e}")

    # Turn the collected rows into a DataFrame
    df = pd.DataFrame(rows)

    # -----------------------------------
    # Force a consistent column order
    # -----------------------------------
    # ordered_cols = [
    #     "Event","Fighter1","Fighter2","Weightclass","Method",
    #     "Round","Time","Format",
    #     "KD1","KD2","SIG_STR1","SIG_STR2","TOTAL_STR1","TOTAL_STR2",
    #     "TD1","TD2","SUB_ATT1","SUB_ATT2","REV1","REV2","CTRL1","CTRL2",
    #     "Head1","Head2","Body1","Body2","Leg1","Leg2",
    #     "Distance1","Distance2","Clinch1","Clinch2","Ground1","Ground2",
    #     "event_date","f1_dob","f1_height","f1_reach","f1_stance",
    #     "f2_dob","f2_height","f2_reach","f2_stance",
    #     "fight_url","f1_url","f2_url"
    # ]
    ordered_cols = [
        "Event","Fighter1","Fighter2",
        "event_date","f1_dob","f1_height","f1_reach","f1_stance",
        "f2_dob","f2_height","f2_reach","f2_stance",
        "fight_url","f1_url","f2_url"
    ]

    df = df[ordered_cols]

    # Save final result
    df.to_csv(out_csv, index=False, encoding="utf-8")
    print(f"\nSaved {len(df)} rows to {out_csv}")

    return df


# -----------------------------------
# Run the scraper
# -----------------------------------
df_new = scrape_and_enrich(
    target_event_name=TARGET_EVENT_NAME,
    before_date=BEFORE_DATE,
    out_csv=OUT_CSV
)

# Quick checks
print(df_new.head())
print(df_new.shape)

Found 1 matching events

Scraping event: UFC Fight Night: Emmett vs. Vallejos | March 14, 2026
  Fight count: 14
    [1/14] http://ufcstats.com/fight-details/f594fc5c2bc6237e
    [2/14] http://ufcstats.com/fight-details/3b7b771b9578383d
    [3/14] http://ufcstats.com/fight-details/8bee5192baec71d8
    [4/14] http://ufcstats.com/fight-details/e0acd0cf4930671b
    [5/14] http://ufcstats.com/fight-details/db7694f4513120d5
    [6/14] http://ufcstats.com/fight-details/fc9168fd102e2e66
    [7/14] http://ufcstats.com/fight-details/f26c0bdf7689aea2
    [8/14] http://ufcstats.com/fight-details/2c330a6f7f1e55f0
    [9/14] http://ufcstats.com/fight-details/f44b4e998b6829f6
    [10/14] http://ufcstats.com/fight-details/c769dd861357af9f
    [11/14] http://ufcstats.com/fight-details/cff81d681a01ae18
    [12/14] http://ufcstats.com/fight-details/8cab057e6549afd0
    [13/14] http://ufcstats.com/fight-details/32d371462c539bee
    [14/14] http://ufcstats.com/fight-details/244285b7fd0ddece

Saved 14 rows